# Notebook 02 — Fine-tuning LoRA (Etapa 2)

O versionamento é uma **função** (`versionar_no_hub`) chamada automaticamente ao fim de cada config.
Você escolhe o tipo de versão (1/2/3) **uma vez, na Config 1**; a **Config 2 herda** essa escolha.
Cada config treina e, ao terminar, **cria a tag sozinha** no Hub.

**Fluxo:** #0 dataset → #1 ambiente + funções → #2 Config 1 (pergunta tipo + treina + versiona) →
#3 Config 2 (herda tipo + treina + versiona) → #4 recuperação (só se OOM) → comparação (Critério 3).

### 0. Verificar o dataset (produzido no notebook 01, no Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/multimodais/dados'
os.environ['DATA_DIR'] = DATA_DIR          # usado nas células de treino via "$DATA_DIR"

meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')
assert os.path.exists(meta_path), f'metadata.jsonl não encontrado em {DATA_DIR}. Rode o notebook 01 primeiro.'
meta = [json.loads(l)['file_name'] for l in open(meta_path, encoding='utf-8')]
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
faltando = [m for m in meta if not os.path.exists(os.path.join(DATA_DIR, m))]
print(f'DATA_DIR = {DATA_DIR} | imagens = {len(imgs)} | metadata = {len(meta)} | faltando = {len(faltando)}')
assert imgs and not faltando, 'Dataset incompleto. Rode/complete o notebook 01 antes de treinar.'


### 1. Ambiente + função de versionamento (rode 1x)

Instala dependências, prepara o script de treino do `diffusers`, faz login no Hub e **define as funções**
`perguntar_tipo()` e `versionar_no_hub()` usadas pelas células de config.

In [ ]:
# torchao NÃO é necessário para LoRA e vinha quebrado (wheel py3.10 em ambiente py3.12) -> remove.
!pip uninstall -y diffusers torchao
!pip -q install "transformers" "accelerate" "datasets" "peft"
![ -d /content/diffusers ] || git clone --depth 1 https://github.com/huggingface/diffusers
!pip -q install -e /content/diffusers
%cd /content/diffusers/examples/text_to_image
!pip -q install -r requirements.txt

import os, re, shutil, torch
from google.colab import userdata
from huggingface_hub import login, HfApi

assert torch.cuda.is_available(), 'SEM GPU. Ambiente de execução -> Alterar tipo -> T4 GPU.'
assert os.environ.get('DATA_DIR'), 'Rode a célula #0 antes (ela define DATA_DIR).'
print('GPU:', torch.cuda.get_device_name(0), '| DATA_DIR:', os.environ['DATA_DIR'])

login(token=userdata.get('HF_TOKEN'))
REPO_ID = 'lamble-lambe/atelie'
api = HfApi(token=userdata.get('HF_TOKEN'))
SEMVER = re.compile(r'^v?(\d+)\.(\d+)\.(\d+)$')

def versao_producao():
    """Maior tag semver no Hub = versão em produção (0.0.0 se não houver)."""
    try:
        tags = [t.name for t in api.list_repo_refs(REPO_ID).tags]
    except Exception:
        tags = []
    vs = [tuple(map(int, m.groups())) for t in tags if (m := SEMVER.match(t))]
    return max(vs) if vs else (0, 0, 0)

def perguntar_tipo(nome):
    """Pergunta o tipo de versão ANTES do treino."""
    print(f'{nome}: que tipo de versão este treino representa?')
    print('  1 = BUG (patch)   2 = FEATURE (minor)   3 = BREAKING (major)')
    return int(input('Digite 1/2/3: ').strip())

def versionar_no_hub(tipo, descricao=''):
    """Incrementa a versão a partir da maior tag e cria a nova tag no último commit do Hub."""
    maj, mnr, pat = versao_producao()
    nova = {1: (maj, mnr, pat + 1), 2: (maj, mnr + 1, 0), 3: (maj + 1, 0, 0)}[tipo]
    tag = f'v{nova[0]}.{nova[1]}.{nova[2]}'
    rotulo = {1: 'BUG/patch', 2: 'FEATURE/minor', 3: 'BREAKING/major'}[tipo]
    api.create_tag(repo_id=REPO_ID, tag=tag, tag_message=descricao or rotulo)
    print(f'\n✅ Versionado: v{maj}.{mnr}.{pat} --({rotulo})--> {tag}')
    print(f'   Hub: https://huggingface.co/{REPO_ID}/tree/{tag}')
    print(f'   Carregar: pipe.load_lora_weights("{REPO_ID}", revision="{tag}")')
    return tag

def subir_e_versionar(output_dir, tipo, descricao=''):
    """Após o treino (SEM --push_to_hub): backup no Drive + upload dos pesos ao Hub + cria a tag.
    Evita o crash do model card do script quando não há --validation_prompt."""
    pesos = os.path.join(output_dir, 'pytorch_lora_weights.safetensors')
    assert os.path.exists(pesos), f'Pesos não encontrados em {pesos} — o treino terminou de verdade?'
    # 1) backup no Drive (sobrevive à queda do runtime)
    backup = '/content/drive/MyDrive/Colab Notebooks/multimodais/backup_lora'
    os.makedirs(backup, exist_ok=True)
    shutil.copy(pesos, os.path.join(backup, os.path.basename(output_dir) + '.safetensors'))
    print('Backup no Drive:', backup)
    # 2) upload ao Hub (manual, sem passar pelo model card bugado)
    api.upload_file(path_or_fileobj=pesos, path_in_repo='pytorch_lora_weights.safetensors',
                    repo_id=REPO_ID, repo_type='model', commit_message=descricao or 'upload dos pesos LoRA')
    # 3) cria a tag
    return versionar_no_hub(tipo, descricao)

print('\nAmbiente pronto. Funções: perguntar_tipo(), subir_e_versionar(), versionar_no_hub().')

### 2. Config 1 — treina, sobe e versiona (rank=8, lr=1e-4, 1500 passos)

Responda o tipo no início (para este dataset novo: **2 = feature**). O treino roda **sem `--push_to_hub`**
(o push do script quebra no model card sem `--validation_prompt`); ao terminar, `subir_e_versionar()`
faz **backup no Drive + upload dos pesos + tag** automaticamente.

In [ ]:
tipo_cfg1 = perguntar_tipo('Config 1')   # <-- responda 1/2/3 ANTES; depois pode se ausentar

# Treino SEM --push_to_hub: o push do script quebra no model card quando não há --validation_prompt
# ("UnboundLocalError: images"). Os pesos são salvos localmente e enviados depois por subir_e_versionar().
!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1500 --learning_rate=1e-4 --lr_scheduler='cosine' \
  --lr_warmup_steps=0 --rank=8 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora-cfg1'

# Treino terminou -> backup no Drive + upload ao Hub + tag (tudo na função).
subir_e_versionar('/content/atelie-lora-cfg1', tipo_cfg1,
                  'Config 1: rank=8, lr=1e-4, 1500 passos, 35 imagens (estilo lambe-lambe)')

### 3. Config 2 — treina e versiona automaticamente (rank=4, lr=5e-5, 1000 passos)

Configuração mais conservadora (menos overfitting) para a comparação do Critério 3.
**Herda o tipo escolhido na Config 1** (mesma natureza de mudança) — não pergunta de novo se a Config 1 já rodou.

In [ ]:
# Herda o tipo escolhido na Config 1 (mesma natureza de mudança).
# Se a Config 1 não foi rodada nesta sessão, pergunta.
tipo_cfg2 = tipo_cfg1 if 'tipo_cfg1' in globals() else perguntar_tipo('Config 2')
print(f'Config 2 usando o mesmo tipo da Config 1: {tipo_cfg2}')

# Treino SEM --push_to_hub (mesma razão da Config 1); upload manual depois.
!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1000 --learning_rate=5e-5 --lr_scheduler='cosine' \
  --lr_warmup_steps=100 --rank=4 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora-cfg2'

subir_e_versionar('/content/atelie-lora-cfg2', tipo_cfg2,
                  'Config 2: rank=4, lr=5e-5, 1000 passos, warmup 100, 35 imagens')

### 4. Recuperação manual (opcional)

As células de config já fazem **backup no Drive + upload + tag** sozinhas. Use esta célula só se precisar
**re-enviar/re-taguear** os pesos a partir de um arquivo — do `/content` (treino da sessão) ou do
**backup no Drive** (`backup_lora/atelie-lora-cfgX.safetensors`), caso tenha perdido o runtime.

In [ ]:
# Recuperação MANUAL — só se, por algum motivo, o upload automático da célula de config não rodou
# (ex.: o treino travou antes, ou você quer re-taguear a partir do backup no Drive).
import os

# Aponte para os pesos: no /content (treino da sessão) OU no backup do Drive (se perdeu o /content).
CAMINHO_PESOS = '/content/atelie-lora-cfg1/pytorch_lora_weights.safetensors'
# Alternativa (perdeu o /content):
# CAMINHO_PESOS = '/content/drive/MyDrive/Colab Notebooks/multimodais/backup_lora/atelie-lora-cfg1.safetensors'
TIPO      = 2                              # 1=bug / 2=feature / 3=breaking
DESCRICAO = 'upload manual dos pesos LoRA'

assert os.path.exists(CAMINHO_PESOS), f'Não encontrei os pesos em {CAMINHO_PESOS}.'
api.upload_file(path_or_fileobj=CAMINHO_PESOS, path_in_repo='pytorch_lora_weights.safetensors',
                repo_id=REPO_ID, repo_type='model', commit_message=DESCRICAO)
versionar_no_hub(TIPO, DESCRICAO)

## Comparação das configurações (Critério 3)

| Hiperparâmetro | Config 1 | Config 2 |
|---|---|---|
| `rank` | 8 | 4 |
| `learning_rate` | 1e-4 | 5e-5 |
| `max_train_steps` | 1500 | 1000 |
| `lr_warmup_steps` | 0 | 100 |
| tag no Hub | `v1.3.0` | `v1.4.0` |

Compare as duas versões via `revision=` no **notebook 03** (grade, CLIPScore, memorização) e justifique
no relatório qual você promoveu para produção (`LORA_REVISION` no Space) e por quê.